# Confidence-Score Exploration

Run all 5 benchmarked models on a set of images and collect their top-1 prediction + confidence into a single wide-format DataFrame.

One row = one image. Columns:

| Column | Content |
|---|---|
| `filename` | image filename |
| `true_class` | parent directory name (ImageFolder convention) |
| `class_<model>` | top-1 predicted class for that model |
| `confidence_<model>` | associated softmax probability |

5 models × 2 cols = 10 prediction columns (+ 2 metadata columns).

Use cases enabled by this shape:
- Mean / median / std of confidence per class per model
- Inter-model agreement rate (fraction of images where all 5 models agree)
- Per-image majority voting
- Disagreement hotspots (which classes cause the models to split)
- Calibration analysis (confidence vs correctness)

**Runs both locally and on Colab** — Cell 1 auto-detects the environment.

## 1. Environment setup (Colab vs local)

In [3]:
# ============================================
# Setup Colab: GPU + Drive
# ============================================

import os
from pathlib import Path

COLAB = "google.colab" in str(get_ipython())

if COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    !nvidia-smi
else:
    print("Running locally")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Wed Apr 15 18:58:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   31C    P0             46W /  600W |       0MiB /  97887MiB |      0%      Default |
|          

In [4]:
# ============================================
# Paths
# ============================================

if COLAB:
    DRIVE_ROOT        = Path("/content/drive/MyDrive/bootcamp_project/data/")
    LOCAL_SSD_ROOT    = Path("/content/dataset/")
    DRIVE_DATASET_DIR = DRIVE_ROOT
    LOCAL_DATASET_DIR = LOCAL_SSD_ROOT
else:
    LOCAL_DATASET_DIR = Path("data/dataset")

SPLIT_DIR  = Path("/content/dataset_split") if COLAB else Path("data/dataset_split")
TRAIN_DIR  = SPLIT_DIR / "train"
VAL_DIR    = SPLIT_DIR / "val"
TEST_DIR   = SPLIT_DIR / "test"   # locked until final evaluation

In [5]:
import shutil

tar_filename = "dataset.tar.gz"
DRIVE_TAR_PATH = DRIVE_DATASET_DIR / tar_filename
LOCAL_TAR_PATH = LOCAL_SSD_ROOT / tar_filename

if COLAB:
    print("\n--- Optimizing Dataset Loading ---")
    print("Checking for dataset tarball on Drive...")
    if not DRIVE_TAR_PATH.exists():
        print(f"Creating tarball '{tar_filename}' from '{DRIVE_DATASET_DIR.name}' in '{DRIVE_DATASET_DIR.parent}'...")
        # Create a tar.gz archive of the dataset directory
        shutil.make_archive(
            str(DRIVE_TAR_PATH.with_suffix('')), # make_archive adds .tar.gz itself
            'gztar',
            root_dir=DRIVE_DATASET_DIR.parent, # The directory containing the folder to archive
            base_dir=DRIVE_DATASET_DIR.name   # The folder to archive
        )
        print(f"✓ Tarball created at {DRIVE_TAR_PATH}")
    else:
        print(f"✓ Tarball already exists at {DRIVE_TAR_PATH}")

    # Check if LOCAL_DATASET_DIR is already populated to skip re-extraction
    # This is useful if a previous run in the same session already extracted the data
    if LOCAL_DATASET_DIR.exists() and any(LOCAL_DATASET_DIR.iterdir()): # Check if directory exists and is not empty
        print("✓ Local dataset directory is already populated. Skipping tarball extraction.")
    else:
        LOCAL_SSD_ROOT.mkdir(parents=True, exist_ok=True)
        print(f"Copying tarball from Drive to local SSD: {DRIVE_TAR_PATH} -> {LOCAL_TAR_PATH}")
        shutil.copy2(DRIVE_TAR_PATH, LOCAL_TAR_PATH)
        print(f"✓ Tarball copied to local SSD.")

        print(f"Extracting tarball to {LOCAL_SSD_ROOT}...")
        shutil.unpack_archive(LOCAL_TAR_PATH, LOCAL_SSD_ROOT)
        print(f"✓ Tarball extracted.")

        # Clean up the copied tarball to save space on SSD
        LOCAL_TAR_PATH.unlink()
        print(f"✓ Local tarball removed: {LOCAL_TAR_PATH}")

    print("--- Dataset loading optimization complete ---")
else:
    print("Local run — no tarball needed.")


--- Optimizing Dataset Loading ---
Checking for dataset tarball on Drive...
✓ Tarball already exists at /content/drive/MyDrive/bootcamp_project/data/dataset.tar.gz
Copying tarball from Drive to local SSD: /content/drive/MyDrive/bootcamp_project/data/dataset.tar.gz -> /content/dataset/dataset.tar.gz
✓ Tarball copied to local SSD.
Extracting tarball to /content/dataset...
✓ Tarball extracted.
✓ Local tarball removed: /content/dataset/dataset.tar.gz
--- Dataset loading optimization complete ---


In [6]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: jimmy-ouellet (jimmy-ouellet-personnal) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [7]:

 """
 Single source of truth for all deployed model configurations.
 Add or comment out entries here to enable/disable models globally.
 """
 from dataclasses import dataclass
 @dataclass(frozen=True)
 class ModelConfig:
     key: str            # identifier used in API requests and wandb artifact names
     timm_name: str      # timm.create_model() identifier
     img_size: int       # native input resolution
     wandb_artifact: str # artifact name in the wandb registry (without :tag)
     enabled: bool = True
 MODEL_REGISTRY: list[ModelConfig] = [
     ModelConfig(
         key="convnext_tiny",
         timm_name="convnext_tiny",
         img_size=224,
         wandb_artifact="convnext_tiny_best",
     ),
     ModelConfig(
         key="efficientnet_b3",
         timm_name="efficientnet_b3",
         img_size=300,
         wandb_artifact="efficientnet_b3_best",
     ),
     ModelConfig(
         key="efficientnet_b4",
         timm_name="efficientnet_b4",
         img_size=380,
         wandb_artifact="efficientnet_b4_best",
     ),
     ModelConfig(
         key="mobilenetv3_large",
         timm_name="mobilenetv3_large_100",
         img_size=224,
         wandb_artifact="mobilenetv3_large_best",
     ),
     ModelConfig(
         key="resnet50",
         timm_name="resnet50",
         img_size=224,
         wandb_artifact="resnet50_best",
     ),
 ]
 # Fast lookup by key
 REGISTRY_BY_KEY: dict[str, ModelConfig] = {m.key: m for m in MODEL_REGISTRY}
 ENABLED_KEYS: list[str] = [m.key for m in MODEL_REGISTRY if m.enabled]

In [8]:
"""
Generic predictor for any timm-compatible classification model.
Replaces the separate model_*.py files - one class handles all 5 architectures.
"""
import pickle
import threading
from pathlib import Path

import timm
import torch
from loguru import logger
from PIL import Image
from torchvision import transforms


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class TimmPredictor:
    """Load a timm model from a wandb artifact and expose predict_top3/predict_set."""

    def __init__(self, cfg: ModelConfig, cache_root: Path | None = None):
        self._cfg        = cfg
        self._cache_root = cache_root
        self._model: torch.nn.Module | None = None
        self._load_error: Exception | None = None
        self._classes: list[str] = []
        self._img_size   = cfg.img_size
        self._ready      = threading.Event()

        self._preprocess = transforms.Compose([
            transforms.Resize((cfg.img_size, cfg.img_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])

    def load(self) -> None:
        """Download artifact (if needed) and load weights into memory."""
        try:
            local_dir = artifact_local_path(self._cfg.wandb_artifact, self._cache_root)

            # Load class names from label encoder or classes.txt
            encoder_files = list(local_dir.glob("*.pkl"))
            if encoder_files:
                with open(encoder_files[0], "rb") as f:
                    le = pickle.load(f)
                self._classes = list(le.classes_)
            else:
                classes_file = local_dir / "classes.txt"
                if classes_file.exists():
                    self._classes = classes_file.read_text().splitlines()
                else:
                    raise FileNotFoundError(
                        f"No label encoder (*.pkl) or classes.txt in {local_dir}. "
                        "Save one alongside the .pth file in the wandb artifact."
                    )

            num_classes = len(self._classes)
            pth_files   = list(local_dir.glob("*.pth"))
            if not pth_files:
                raise FileNotFoundError(f"No .pth file found in {local_dir}")

            self._model = timm.create_model(
                self._cfg.timm_name, pretrained=False, num_classes=num_classes
            )
            self._model.load_state_dict(torch.load(pth_files[0], map_location=DEVICE))
            self._model.to(DEVICE)
            self._model.train(False)   # sets inference mode (same as .eval())
            logger.info("{} ready. device={} classes={}",
                        self._cfg.key, DEVICE, num_classes)
        except Exception as exc:
            self._load_error = exc
            logger.error("Failed to load {}: {}", self._cfg.key, exc)
        finally:
            self._ready.set()

    def _check_ready(self) -> None:
        """Block until load() completes, then raise if it failed."""
        self._ready.wait()
        if self._load_error is not None:
            raise RuntimeError(
                f"Model {self._cfg.key} failed to load"
            ) from self._load_error

    def predict_top3(self, img_path: str) -> list[tuple[str, float]]:
        """Return top-3 (class_name, confidence) for a single image."""
        self._check_ready()
        tensor = self._preprocess(
            Image.open(img_path).convert("RGB")
        ).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            proba = torch.softmax(self._model(tensor), dim=1).squeeze()
        k = min(3, len(self._classes))
        top3 = proba.topk(k)
        return [
            (self._classes[i.item()], round(p.item(), 4))
            for i, p in zip(top3.indices, top3.values)
        ]

    def predict_set(self, img_paths: list[str],
                    batch_size: int = 32) -> list[tuple[str, float]]:
        """Return top-1 (class_name, confidence) for each image in a batch."""
        self._check_ready()
        results = []
        for start in range(0, len(img_paths), batch_size):
            chunk = img_paths[start: start + batch_size]
            batch = torch.stack([
                self._preprocess(Image.open(p).convert("RGB")) for p in chunk
            ]).to(DEVICE)
            with torch.no_grad():
                proba = torch.softmax(self._model(batch), dim=1)
            confs, idxs = proba.max(dim=1)
            results.extend(
                (self._classes[i.item()], round(c.item(), 4))
                for i, c in zip(idxs, confs)
            )
        return results


In [9]:
"""
Download wandb model artifacts to a local cache directory.
Mirrors the TTL-based pattern from the old gcs_cache.py.
"""
import os
import shutil
import tempfile
import time
from pathlib import Path

from loguru import logger

_CACHE_MAX_AGE = int(os.getenv("WANDB_CACHE_MAX_AGE_SECONDS", str(3 * 3600)))
_WANDB_PROJECT = os.getenv("WANDB_PROJECT", "certification")
_WANDB_ENTITY  = os.getenv("WANDB_ENTITY", "")


def is_cache_valid(local_dir: Path, filenames: list[str],
                   max_age: int = _CACHE_MAX_AGE) -> bool:
    """Return True if all files exist and are fresher than max_age seconds."""
    if not local_dir.exists():
        return False
    now = time.time()
    for name in filenames:
        p = local_dir / name
        if not p.exists():
            logger.debug("Cache miss: {} not found.", p)
            return False
        if now - p.stat().st_mtime > max_age:
            logger.debug("Cache expired: {}.", p)
            return False
    logger.info("Cache hit for {}.", local_dir)
    return True


def artifact_local_path(artifact_name: str, cache_root: Path | None = None) -> Path:
    """
    Return a local directory containing the artifact files.
    Downloads from wandb if not cached; falls back to a .pth in cwd.

    Resolution order:
    1. Local cache (TTL-based) - skip download if a fresh copy exists
    2. wandb registry download
    3. Local fallback - look for {artifact_name}.pth in cwd
    """
    if cache_root is None:
        cache_root = Path.cwd() / "models" / "wandb"

    # Validate artifact_name to prevent path traversal attacks
    safe_name = Path(artifact_name).name
    if not safe_name or safe_name != artifact_name:
        raise ValueError(f"artifact_name must be a plain name, got: {artifact_name!r}")

    local_dir = cache_root / safe_name
    pth_files = list(local_dir.glob("*.pth")) if local_dir.exists() else []

    if pth_files and is_cache_valid(local_dir, [pth_files[0].name]):
        return local_dir

    logger.info("Downloading wandb artifact: {}", artifact_name)
    try:
        import wandb
        entity_prefix = f"{_WANDB_ENTITY}/" if _WANDB_ENTITY else ""
        ref = f"{entity_prefix}{_WANDB_PROJECT}/{artifact_name}:latest"
        run = wandb.init(project=_WANDB_PROJECT, job_type="inference",
                         reinit="finish_previous")
        try:
            artifact = run.use_artifact(ref, type="model")

            # Download to temp dir first to prevent cache poisoning if download fails
            tmp_dir = cache_root / f"{safe_name}.tmp"
            shutil.rmtree(tmp_dir, ignore_errors=True)
            tmp_dir.mkdir(parents=True, exist_ok=True)
            artifact.download(root=str(tmp_dir))

            # Atomically replace the destination directory
            if local_dir.exists():
                shutil.rmtree(local_dir)
            tmp_dir.rename(local_dir)

            logger.info("Downloaded to {}.", local_dir)
        finally:
            run.finish()
    except Exception as exc:
        logger.warning("wandb download failed ({}). Trying local fallback.", exc)
        fallback = Path.cwd() / f"{artifact_name}.pth"
        if fallback.exists():
            local_dir.mkdir(parents=True, exist_ok=True)
            shutil.copy2(fallback, local_dir / fallback.name)
            logger.info("Using local fallback: {}.", fallback)
        else:
            raise FileNotFoundError(
                f"wandb download failed and no local fallback found for {artifact_name}. "
                f"Place {artifact_name}.pth in the working directory or set WANDB_API_KEY."
            ) from exc

    return local_dir


In [10]:
import random
import threading

import pandas as pd
import torch
from tqdm.auto import tqdm

print('Device        :', 'cuda' if torch.cuda.is_available() else 'cpu')
print('Models to run :', ENABLED_KEYS)

Device        : cuda
Models to run : ['convnext_tiny', 'efficientnet_b3', 'efficientnet_b4', 'mobilenetv3_large', 'resnet50']


In [11]:
! pip install -q timm

## 2. Load all 5 models in parallel

`TimmPredictor.load()` pulls each artifact from wandb (or falls back to a local `<artifact>.pth`) and blocks predict calls on a `threading.Event` until weights are in memory. We load in parallel to shave startup time.

In [13]:
# Where wandb artifacts get cached. On Colab we keep them on the local SSD (fast)
# rather than Drive (slow). They will be lost when the runtime recycles, but
# re-downloading from wandb is quick on a good connection.
CACHE_ROOT = Path('/content/wandb_cache') if IN_COLAB else ("/content/wandb")
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

predictors: dict[str, TimmPredictor] = {}
threads = []

for cfg in MODEL_REGISTRY:
    if not cfg.enabled:
        continue
    p = TimmPredictor(cfg, cache_root=CACHE_ROOT)
    predictors[cfg.key] = p
    t = threading.Thread(target=p.load, daemon=True)
    t.start()
    threads.append(t)

for t in threads:
    t.join()

for key, p in predictors.items():
    if p._load_error is not None:
        raise RuntimeError(f'{key} failed to load') from p._load_error

print('All models ready:', list(predictors))

2026-04-15 19:08:20.913 | INFO     | __main__:is_cache_valid:32 - Cache hit for /content/wandb_cache/convnext_tiny_best.
2026-04-15 19:08:20.914 | INFO     | __main__:is_cache_valid:32 - Cache hit for /content/wandb_cache/efficientnet_b4_best.
2026-04-15 19:08:20.915 | INFO     | __main__:is_cache_valid:32 - Cache hit for /content/wandb_cache/mobilenetv3_large_best.
2026-04-15 19:08:20.915 | INFO     | __main__:is_cache_valid:32 - Cache hit for /content/wandb_cache/efficientnet_b3_best.
2026-04-15 19:08:20.915 | INFO     | __main__:is_cache_valid:32 - Cache hit for /content/wandb_cache/resnet50_best.
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
2

All models ready: ['convnext_tiny', 'efficientnet_b3', 'efficientnet_b4', 'mobilenetv3_large', 'resnet50']


## 3. Collect image paths

Point `DATA_ROOT` at an `ImageFolder`-style directory (one subdirectory per class). Set `SAMPLE_SIZE` to an integer for a random subset, or `None` to use everything.

On Colab the dataset typically lives on Google Drive or in a tarball you have extracted to `/content`. Extract before running this cell if needed:

```python
# example: one-time extraction from a Drive tarball to local SSD
# !tar -xzf /content/drive/MyDrive/plant_detect/dataset.tar.gz -C /content/
```

In [14]:
if IN_COLAB:
    DATA_ROOT = Path('/content/dataset/')               # adjust
    OUTPUT_DIR = Path('/content/drive/MyDrive/bootcamp_project/figures')
else:
    DATA_ROOT = REPO_ROOT / 'data' / 'raw' / 'all_images'  # adjust
    OUTPUT_DIR = REPO_ROOT / 'figures'

SAMPLE_SIZE: int | None = 25000  # None = use every image
RANDOM_SEED = 42
VALID_EXT   = {'.jpg', '.jpeg', '.png'}

all_paths = [
    p for p in DATA_ROOT.rglob('*')
    if p.is_file() and p.suffix.lower() in VALID_EXT
]

if SAMPLE_SIZE is not None and SAMPLE_SIZE < len(all_paths):
    random.seed(RANDOM_SEED)
    all_paths = random.sample(all_paths, SAMPLE_SIZE)

all_paths.sort()  # deterministic row order
print(f'{len(all_paths)} images from {DATA_ROOT}')

25000 images from /content/dataset


## 4. Run batch prediction for each model

`predict_set` already batches internally. On a Colab T4/L4 GPU, batch_size 128–256 works comfortably for the smaller backbones; reduce if you see out-of-memory errors on EfficientNet-B4 (380×380 inputs).

In [15]:
BATCH_SIZE = 128 if torch.cuda.is_available() else 32

path_strs = [str(p) for p in all_paths]
results_per_model: dict[str, list[tuple[str, float]]] = {}

for key, p in tqdm(predictors.items(), desc='models'):
    results_per_model[key] = p.predict_set(path_strs, batch_size=BATCH_SIZE)

print('Done.')

models:   0%|          | 0/5 [00:00<?, ?it/s]

Done.


## 5. Assemble the DataFrame

In [16]:
rows = {
    'filename'  : [p.name for p in all_paths],
    'true_class': [p.parent.name for p in all_paths],
}
for key, preds in results_per_model.items():
    rows[f'class_{key}']      = [cls for cls, _ in preds]
    rows[f'confidence_{key}'] = [conf for _, conf in preds]

df = pd.DataFrame(rows)
df.head()

,filename,true_class,class_convnext_tiny,confidence_convnext_tiny,class_efficientnet_b3,confidence_efficientnet_b3,class_efficientnet_b4,confidence_efficientnet_b4,class_mobilenetv3_large,confidence_mobilenetv3_large,class_resnet50,confidence_resnet50
0,allium_0003.jpg,allium,allium,0.8742,allium,0.8184,allium,0.6836,allium,0.7490,allium,0.8374
1,allium_0006.jpg,allium,allium,0.9015,allium,0.7993,allium,0.6488,allium,0.7981,allium,0.8848
2,allium_0007.jpg,allium,allium,0.9351,allium,0.8442,allium,0.7511,allium,0.8914,lily,0.4908
3,allium_0022.jpg,allium,allium,0.8726,allium,0.8949,allium,0.9247,allium,0.8795,allium,0.8674
4,allium_0026.jpg,allium,allium,0.8767,allium,0.9340,allium,0.8770,allium,0.6077,allium,0.8052


In [17]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV = OUTPUT_DIR / 'confidence_exploration.csv'
df.to_csv(OUTPUT_CSV, index=False)
print('Saved', OUTPUT_CSV)

Saved /content/drive/MyDrive/bootcamp_project/figures/confidence_exploration.csv


## 6. Example analyses

Starters — run, adapt, replace.

### 6.1 Mean confidence per class per model

In [28]:
OUTPUT_CSV = OUTPUT_DIR / 'correct_vs_incorrect_confidence.csv'
test.to_csv(OUTPUT_CSV, index=False)

In [18]:
conf_cols = [f'confidence_{k}' for k in predictors]
(df.groupby('true_class')[conf_cols]
   .mean()
   .round(3)
   .sort_values(conf_cols[0]))

,confidence_convnext_tiny,confidence_efficientnet_b3,confidence_efficientnet_b4,confidence_mobilenetv3_large,confidence_resnet50
true_class,,,,,
chives,0.803,0.762,0.778,0.695,0.725
thyme,0.828,0.774,0.780,0.730,0.673
blackberry,0.842,0.789,0.757,0.657,0.562
apple,0.844,0.806,0.771,0.696,0.543
lavender,0.844,0.802,0.781,0.729,0.734
mint,0.845,0.791,0.776,0.684,0.662
poppy,0.847,0.830,0.818,0.781,0.732
dill,0.848,0.784,0.811,0.772,0.751
chamomile,0.848,0.816,0.823,0.746,0.750


### 6.2 Correct vs incorrect confidence (calibration signal)

In [24]:
summary = []
for key in predictors:
    correct = df[f'class_{key}'] == df['true_class']
    summary.append({
        'model'        : key,
        'accuracy'     : correct.mean(),
        'conf_correct' : df.loc[correct, f'confidence_{key}'].mean(),
        'conf_wrong'   : df.loc[~correct, f'confidence_{key}'].mean(),
    })
test = pd.DataFrame(summary).round(3)

### 6.3 Inter-model agreement (row level)

In [20]:
class_cols = [f'class_{k}' for k in predictors]
df['n_distinct_predictions'] = df[class_cols].nunique(axis=1)

agreement = df['n_distinct_predictions'].value_counts().sort_index()
agreement_pct = (agreement / len(df) * 100).round(1)
pd.DataFrame({'n_images': agreement, 'pct': agreement_pct})

,n_images,pct
n_distinct_predictions,,
1,22245,89.0
2,2215,8.9
3,415,1.7
4,105,0.4
5,20,0.1


### 6.4 Pairwise agreement matrix

In [21]:
import numpy as np

keys = list(predictors)
agree = np.zeros((len(keys), len(keys)))
for i, a in enumerate(keys):
    for j, b in enumerate(keys):
        agree[i, j] = (df[f'class_{a}'] == df[f'class_{b}']).mean()
pd.DataFrame(agree, index=keys, columns=keys).round(3)

,convnext_tiny,efficientnet_b3,efficientnet_b4,mobilenetv3_large,resnet50
convnext_tiny,1.000,0.972,0.971,0.953,0.909
efficientnet_b3,0.972,1.000,0.981,0.962,0.913
efficientnet_b4,0.971,0.981,1.000,0.962,0.914
mobilenetv3_large,0.953,0.962,0.962,1.000,0.912
resnet50,0.909,0.913,0.914,0.912,1.000


### 6.5 Hardest cases (≥3 distinct model predictions)

In [22]:
hard = df[df['n_distinct_predictions'] >= 3].copy()
hard['mean_confidence'] = hard[conf_cols].mean(axis=1).round(3)
hard.sort_values('mean_confidence').head(20)

,filename,true_class,class_convnext_tiny,confidence_convnext_tiny,class_efficientnet_b3,confidence_efficientnet_b3,class_efficientnet_b4,confidence_efficientnet_b4,class_mobilenetv3_large,confidence_mobilenetv3_large,class_resnet50,confidence_resnet50,n_distinct_predictions,mean_confidence
6459,cosmos_1022.jpg,cosmos,wisteria,0.1005,gypsophila,0.1413,dill,0.1108,gerbera,0.2291,strawberry,0.2186,5,0.160
24220,wisteria_0538.jpg,wisteria,allium,0.1333,lily,0.1727,foxglove,0.1895,iris,0.1361,lily,0.2966,4,0.186
10395,grape_1082.jpg,grape,grape,0.4129,blackberry,0.1723,avocado,0.1424,grape,0.1362,avocado,0.1779,3,0.208
10059,gerbera_1307.jpg,gerbera,foxglove,0.3132,melon,0.1547,gerbera,0.1834,oregano,0.2124,fig,0.2003,5,0.213
14475,lemon_verbena_20269.jpg,lemonverbena,lemonverbena,0.2259,lemonverbena,0.1513,lemonverbena,0.3228,mango,0.1551,peach,0.2127,3,0.214
377,allium_1403.jpg,allium,angelica,0.2202,angelica,0.1905,sage,0.1843,rosemary,0.4243,tarragon,0.1161,4,0.227
17614,mugwort_84.jpg,mugwort,mugwort,0.3415,blackberry,0.1657,basil,0.2406,thyme,0.1072,pear,0.3049,5,0.232
9126,foxglove_1056.jpg,foxglove,fennel,0.2574,fig,0.1048,blackberry,0.2442,fig,0.2832,fennel,0.3199,3,0.242
5721,chrysanthemum_1244.jpg,chrysanthemum,ranunculus,0.4259,oregano,0.1814,ranunculus,0.2282,angelica,0.2274,angelica,0.1565,3,0.244
11683,hydrangea_1111.jpg,hydrangea,hydrangea,0.3311,mango,0.1735,pear,0.2194,allium,0.1606,wintergreen,0.3351,5,0.244
